# 🟠 05 — Cosmos as Tools: Building Blocks You Snap Together

> **Goal:** learn that every Cosmos capability is also a **tool** — a plain function you
> can call directly, *or* hand to any agent (even a non-Cosmos one).

---

## Two ways to use a tool

```mermaid
flowchart TD
    subgraph A["Way 1 — call it directly"]
        direction LR
        F["🟠 video_probe(path)"]:::tool --> R1([📄 result dict]):::out
    end
    subgraph B["Way 2 — give it to an agent"]
        direction LR
        AG["🤖 any Agent<br/>(Bedrock, OpenAI, Ollama…)"]:::agent
        T["🟠 tools=[cosmos_vision_invoke]"]:::tool
        AG --> T --> R2([📄 the agent calls it<br/>when it needs to]):::out
    end
    classDef user fill:#ECEFF1,stroke:#607D8B,stroke-width:2px,color:#263238
    classDef agent fill:#E3F2FD,stroke:#1976D2,stroke-width:2px,color:#0D47A1
    classDef reason fill:#E8F5E9,stroke:#388E3C,stroke-width:2px,color:#1B5E20
    classDef gen fill:#F3E5F5,stroke:#8E24AA,stroke-width:2px,color:#4A148C
    classDef tool fill:#FFF3E0,stroke:#F57C00,stroke-width:2px,color:#E65100
    classDef data fill:#FFFDE7,stroke:#FBC02D,stroke-width:2px,color:#F57F17
    classDef out fill:#E0F2F1,stroke:#00897B,stroke-width:2px,color:#004D40
    classDef think fill:#FCE4EC,stroke:#D81B60,stroke-width:2px,color:#880E4F
```

Many tools (like `video_probe`, `cosmos_sysinfo`, `image_read`) need **no GPU** at all —
they're just helpful utilities.

In [ ]:
# 🔧 Preflight — this cell never crashes. It just tells us what's available.
import os, sys, time
os.environ.setdefault("QWEN_VL_VIDEO_READER", "torchcodec")
sys.path.insert(0, os.path.abspath(".."))

# 🔒 Cosmos tools confine file access to a workspace allow-list for safety.
# The notebooks live in notebooks/ but our sample media sits one level up, so we
# explicitly widen the workspace to the project root + /tmp. (This is how you grant
# the tools access to a folder — never wider than you need.)
os.environ["COSMOS_WORKSPACE"] = os.pathsep.join([os.path.abspath(".."), "/tmp"])            # import strands_cosmos from repo root

PROJECT_ROOT = os.path.abspath("..")
SAMPLE_VIDEO = os.path.join(PROJECT_ROOT, "sample.mp4")
SAMPLE_IMAGE = os.path.join(PROJECT_ROOT, "sample.png")

def gpu_ready() -> bool:
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

READY = gpu_ready()
print("🎮 GPU available :", READY)
if READY:
    import torch
    print("   Device       :", torch.cuda.get_device_name(0))
    print(f"   VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("   (CPU-only is fine for reading along — the heavy cells will skip safely.)")
print("📹 sample.mp4   :", os.path.exists(SAMPLE_VIDEO))
print("🖼  sample.png   :", os.path.exists(SAMPLE_IMAGE))

## Tool 1 — `cosmos_sysinfo`: what hardware am I on?
Great first call on any new machine. Works everywhere.

In [ ]:
from strands_cosmos import cosmos_sysinfo
print(cosmos_sysinfo()["content"][0]["text"])

## Tool 2 — `video_probe`: inspect a video (no GPU)

In [ ]:
from strands_cosmos import video_probe
if os.path.exists(SAMPLE_VIDEO):
    r = video_probe(video_path=SAMPLE_VIDEO)
    print("status:", r["status"])
    print(r["content"][0]["text"][:400])
else:
    print("⏭  sample.mp4 not found.")

## Tool 3 — `video_extract_frames`: pull snapshots out of a video
Returns the output folder and embeds the first frame so you can see it.

In [ ]:
from strands_cosmos import video_extract_frames
if os.path.exists(SAMPLE_VIDEO):
    r = video_extract_frames(video_path=SAMPLE_VIDEO, fps=1.0, max_frames=3)
    print("status:", r["status"])
    print(r["content"][0]["text"])
else:
    print("⏭  sample.mp4 not found.")

## Tool 4 — `image_read`: load an image so an agent can *see* it
Confined to the project workspace for safety, then embedded in the result.

In [ ]:
from strands_cosmos import image_read
if os.path.exists(SAMPLE_IMAGE):
    r = image_read(image_path=SAMPLE_IMAGE)
    print("status:", r["status"], "— image is now embedded for the agent to view")
else:
    print("⏭  sample.png not found.")

## Tool 5 — `cosmos_vision_invoke`: full video understanding in one call
This one **does** need a GPU. It's the all-in-one "describe this video" tool — the same
power as notebook 02, but as a single function you can drop into any agent.

In [ ]:
from strands_cosmos import cosmos_vision_invoke
if READY and os.path.exists(SAMPLE_VIDEO):
    t0 = time.time()
    r = cosmos_vision_invoke(prompt="Describe the scene briefly.",
                             video_path=SAMPLE_VIDEO, max_tokens=512)
    print("status:", r["status"], f"({time.time()-t0:.1f}s)")
    if r["status"] == "success":
        print(r["content"][0]["text"][:300], "...")
else:
    print("⏭  Need a GPU + sample.mp4 for this one.")

## Bonus — hand the tool to *any* agent
Here's the pattern. The agent decides **when** to call Cosmos on its own:

```python
from strands import Agent
from strands_cosmos import cosmos_vision_invoke, video_probe

# Works with ANY model provider — Bedrock, OpenAI, Ollama, …
agent = Agent(tools=[cosmos_vision_invoke, video_probe])
agent('Probe sample.mp4, then describe what happens in it.')
```

# 🎓 What you learned

| Tool | Needs GPU? | Does |
|------|:---------:|------|
| `cosmos_sysinfo` | ❌ | Reports your hardware |
| `video_probe` | ❌ | Video metadata |
| `video_extract_frames` | ❌ | Frames → JPEGs |
| `image_read` | ❌ | Load image for an agent to see |
| `cosmos_vision_invoke` | ✅ | Full video understanding |

Two ways to use any of them: **call directly**, or **`tools=[...]` on an agent**.

**Next:** [06 — Cosmos 3: Understand](06_cosmos3_understand.ipynb) — step up to the newest,
most powerful model. 🌟 →